# 00 · Data acquisition & provenance

This notebook downloads and caches **every** real, open-access dataset used in the
rapamycin multi-omics series, and records its provenance. Run it first.

**Layers & sources**

| Layer | Dataset | Repository |
|-------|---------|-----------|
| Target / mechanism | Rapamycin → FKBP1A/MTOR | ChEMBL (CHEMBL413) |
| Transcriptomics | GSE131754 mouse liver RNA-seq (rapamycin vs control) | NCBI GEO — Tyshkovskiy *et al.*, *Cell Metab* 2019 |
| Pharmacogenomics | GDSC1 Temsirolimus IC50 (~970 cancer cell lines) | Therapeutics Data Commons |
| Proteomics | PXD067812 rapamycin-treated DLBCL, TMT-18plex global + phospho | PRIDE/ProteomeXchange |

All data are public. Nothing here constitutes clinical efficacy evidence; these are
secondary analyses of published molecular data.

In [1]:
import os, sys, json, gzip, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Resolve project root whether run from notebooks/ or project root.
CWD = Path.cwd()
PROJ = CWD.parent if CWD.name == "notebooks" else CWD
RAW = PROJ / "data" / "raw"
PROC = PROJ / "data" / "processed"
FIG = PROJ / "figures"
for d in (RAW, PROC, FIG):
    d.mkdir(parents=True, exist_ok=True)

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")
print("Project root:", PROJ)

Project root: /Users/jinmo/Documents/GitHub/Series1/Jin001_Docking/rapamycin_multiomics


## 1 · Transcriptomics — GEO GSE131754 (rapamycin vs control RNA-seq)

In [2]:
import urllib.request
GSE_URL = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/"
           "GSE131754/suppl/GSE131754_Interventions_assigned_reads.txt.gz")
gse_path = RAW / "GSE131754_Interventions_assigned_reads.txt.gz"
if not gse_path.exists():
    print("Downloading GSE131754 count matrix ...")
    urllib.request.urlretrieve(GSE_URL, gse_path)
counts = pd.read_csv(gse_path, sep="\t", index_col=0)
print("RNA-seq counts:", counts.shape, "(genes x samples)")
rap = [c for c in counts.columns if c.startswith("RAP")]
con = [c for c in counts.columns if c.startswith("CON")]
print(f"Rapamycin samples: {len(rap)} | Control samples: {len(con)}")
counts.iloc[:3, :6]

RNA-seq counts: (43629, 78) (genes x samples)
Rapamycin samples: 12 | Control samples: 12


,ACA_12m_F_1,ACA_12m_F_2,ACA_12m_F_3,ACA_12m_M_1,ACA_12m_M_2,ACA_12m_M_3
GENE_ID,,,,,,
ENSMUSG00000102693,0,0,0,0,0,0
ENSMUSG00000064842,0,0,0,0,0,0
ENSMUSG00000051951,0,0,0,0,2,0


## 2 · Proteomics — PRIDE PXD067812 (rapamycin-treated DLBCL, TMT-18plex)

In [3]:
MZTAB_URL = ("https://ftp.pride.ebi.ac.uk/pride/data/archive/2025/09/"
             "PXD067812/P1426-TMT18-10-F.mzTab")
mztab_path = RAW / "PXD067812.mzTab"
if not mztab_path.exists():
    print("Downloading PXD067812 mzTab (~85 MB) ...")
    urllib.request.urlretrieve(MZTAB_URL, mztab_path)
print("mzTab cached:", round(mztab_path.stat().st_size / 1e6, 1), "MB")

# TMT-18plex channel -> condition map (from PRIDE 'Table 1')
TMT_MAP = {
    1: ("DMSO", "DMSO_1"), 2: ("DMSO", "DMSO_2"), 3: ("DMSO", "DMSO_3"),
    4: ("Rapamycin_24h", "Rapa24h_1"), 5: ("Rapamycin_24h", "Rapa24h_2"), 6: ("Rapamycin_24h", "Rapa24h_3"),
    7: ("Rapamycin_48h", "Rapa48h_1"), 8: ("Rapamycin_48h", "Rapa48h_2"), 9: ("Rapamycin_48h", "Rapa48h_3"),
    10: ("MasterMix", "MasterMix"),
}
pd.DataFrame([(k, *v) for k, v in TMT_MAP.items()],
             columns=["assay", "condition", "sample"]).to_csv(PROC / "proteomics_channel_map.csv", index=False)
print("Saved channel map -> proteomics_channel_map.csv")

mzTab cached: 85.0 MB
Saved channel map -> proteomics_channel_map.csv


## 3 · Pharmacogenomics — GDSC1 drug response via Therapeutics Data Commons

In [4]:
from tdc.multi_pred import DrugRes
gdsc = DrugRes(name="GDSC1", path=str(RAW))
df = gdsc.get_data()
print("GDSC1:", df.shape, "| drugs:", df['Drug_ID'].nunique(), "| cell lines:", df['Cell Line_ID'].nunique())
# Temsirolimus = CCI-779, a rapamycin ester prodrug / mTOR inhibitor
tem = df[df['Drug_ID'].astype(str).str.contains('Temsirolimus', case=False)].copy()
tem.to_csv(PROC / "gdsc_temsirolimus.csv", index=False)
print("Temsirolimus response rows:", tem.shape[0])
tem.head()

Downloading...


  0%|          | 0.00/140M [00:00<?, ?iB/s]

  0%|          | 17.4k/140M [00:00<20:55, 112kiB/s]

  0%|          | 69.6k/140M [00:00<10:59, 213kiB/s]

  0%|          | 174k/140M [00:00<06:19, 370kiB/s] 

...

100%|██████████| 140M/140M [00:38<00:00, 3.61MiB/s]


Loading...


Done!


GDSC1: (177310, 5) | drugs: 208 | cell lines: 958
Temsirolimus response rows: 911


,Drug_ID,Drug,Cell Line_ID,Cell Line,Y
112800,Temsirolimus,C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[...,MC-CAR,"[3.23827250519154, 2.98225419469807, 10.235490...",-3.957970
112801,Temsirolimus,C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[...,PFSK-1,"[7.78071324341182, 2.75325311612598, 9.9601374...",-4.281551
112802,Temsirolimus,C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[...,A673,"[7.301344403901299, 2.89053310182054, 9.922489...",-2.361187
112803,Temsirolimus,C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[...,ES3,"[8.690197905033282, 3.0914731119366, 9.9924871...",-3.289580
112804,Temsirolimus,C[C@@H]1CC[C@H]2C[C@@H](/C(=C/C=C/C=C/[C@H](C[...,ES5,"[8.233101127037282, 2.82468731112752, 10.01588...",-3.488040


## 4 · Target / mechanism — ChEMBL (downloaded live in notebook 01)

In [5]:
import requests
r = requests.get("https://www.ebi.ac.uk/chembl/api/data/molecule/CHEMBL413.json", timeout=60)
mol = r.json()
print("ChEMBL molecule:", mol.get("pref_name"), "| CHEMBL413")
print("Max phase (approval):", mol.get("max_phase"))

ChEMBL molecule: SIROLIMUS | CHEMBL413
Max phase (approval): 4.0


## 5 · Provenance summary

In [6]:
provenance = pd.DataFrame([
    dict(layer="Target/mechanism", dataset="ChEMBL CHEMBL413 (rapamycin/sirolimus)",
         source="EMBL-EBI ChEMBL v34", access="REST API"),
    dict(layer="Transcriptomics", dataset="GSE131754 mouse liver RNA-seq (RAP=12 vs CON=12)",
         source="NCBI GEO; Tyshkovskiy 2019 Cell Metab 10.1016/j.cmet.2019.06.018", access="GEO FTP"),
    dict(layer="Pharmacogenomics", dataset="GDSC1 Temsirolimus IC50",
         source="Therapeutics Data Commons (DrugRes)", access="PyTDC"),
    dict(layer="Proteomics", dataset="PXD067812 SU-DHL-4 rapamycin TMT-18plex",
         source="PRIDE/ProteomeXchange", access="PRIDE FTP (mzTab)"),
])
provenance.to_csv(PROC / "provenance.csv", index=False)
provenance

,layer,dataset,source,access
0,Target/mechanism,ChEMBL CHEMBL413 (rapamycin/sirolimus),EMBL-EBI ChEMBL v34,REST API
1,Transcriptomics,GSE131754 mouse liver RNA-seq (RAP=12 vs CON=12),NCBI GEO; Tyshkovskiy 2019 Cell Metab 10.1016/...,GEO FTP
2,Pharmacogenomics,GDSC1 Temsirolimus IC50,Therapeutics Data Commons (DrugRes),PyTDC
3,Proteomics,PXD067812 SU-DHL-4 rapamycin TMT-18plex,PRIDE/ProteomeXchange,PRIDE FTP (mzTab)


**Done.** All datasets are cached under `data/raw/` and key extracts under `data/processed/`.
Proceed to `01_target_mechanism.ipynb`.